# 04 - Station verification

The comparison behind Figure S1 and Table S1: a gridded product against point
observations. Two caveats govern how the numbers should be read, and both are
about the observations rather than the model.

**Representativeness.** A grid cell is an area average, a station is a point. At
2.5 km over Alpine terrain the two can differ by hundreds of metres in
elevation, and the resulting offset is often larger than the model error being
measured.

**Coverage.** Roughly 90 % of Alpine stations lie below 1,500 m -- exactly where
the terrain is least demanding. Aggregate statistics over such a network
flatter every product.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

from alpinemet.verification import (
    elevation_band_metrics,
    extract_at_stations,
    lapse_rate_adjustment,
    match_stations_to_grid,
    station_metrics,
    verification_metrics,
)

## A synthetic 2.5 km product and a station network

In [ ]:
time = pd.date_range("2020-01-01", periods=24 * 60, freq="h")
lats = np.arange(46.5, 48.01, 0.025)
lons = np.arange(10.0, 14.01, 0.025)

rng = np.random.default_rng(5)

# Model orography: a smooth ridge running west to east.
orography = 800.0 + 1200.0 * np.exp(
    -(((lats[:, None] - 47.3) / 0.35) ** 2) - ((lons[None, :] - 12.0) / 2.0) ** 2
)

seasonal = 2.0 - 8.0 * np.cos(2 * np.pi * (time.dayofyear.to_numpy() - 15) / 365)
field = seasonal[:, None, None] - 0.0065 * orography[None, :, :]
field = field + rng.normal(0.0, 0.8, field.shape)

model = xr.Dataset(
    {
        "temperature_2m": xr.DataArray(
            field,
            dims=("time", "latitude", "longitude"),
            coords={"time": time, "latitude": lats, "longitude": lons},
            attrs={"units": "degC"},
        ),
        "orography": xr.DataArray(
            orography,
            dims=("latitude", "longitude"),
            coords={"latitude": lats, "longitude": lons},
        ),
    }
)

stations = pd.DataFrame(
    {
        "synnr": [11399, 11200, 11111, 11146, 11343],
        "name": ["TAMSWEG", "KALS", "TANNHEIM", "OBERGURGL", "SEEFELD"],
        "lon": [13.808, 12.646, 10.506, 11.024, 11.190],
        "lat": [47.133, 47.005, 47.500, 46.867, 47.329],
        "alt": [1025.0, 1352.0, 1100.0, 1938.0, 1180.0],
    }
)
stations

## Match each station to its grid cell

In [ ]:
matched = match_stations_to_grid(model, stations, elevation_variable="orography")
matched[["name", "alt", "grid_elevation", "elevation_difference", "distance_km"]]

The elevation difference is the quantity to inspect before believing any bias.
Where the model cell sits well above its station, a cold bias is expected and
says more about representativeness than about model skill.

## Extract the modelled series and pair it with observations

In [ ]:
modelled = extract_at_stations(model["temperature_2m"], matched)

# Synthetic observations: the true station-elevation temperature plus noise.
elevation_by_station = dict(zip(stations["synnr"], stations["alt"], strict=True))
grid_elevation = dict(zip(matched["synnr"], matched["grid_elevation"], strict=True))

paired = modelled.rename(columns={"value": "modelled"})
paired["station_elevation"] = paired["station_id"].map(elevation_by_station)
paired["grid_elevation"] = paired["station_id"].map(grid_elevation)
paired["elevation_difference"] = paired["grid_elevation"] - paired["station_elevation"]
paired["observed"] = paired["modelled"] - 0.0065 * paired["elevation_difference"]
paired["observed"] += rng.normal(0.0, 0.5, len(paired))

paired.head()

## Metrics per station

In [ ]:
per_station = station_metrics(paired, station_metadata=stations)
per_station[["name", "alt", "n_pairs", "bias", "rmse", "correlation"]]

The bias tracks the elevation offset almost exactly, which is the point: without
inspecting it, one would report these as model errors.

## Correcting for the elevation offset

In [ ]:
paired["adjusted"] = lapse_rate_adjustment(
    paired["modelled"], paired["elevation_difference"]
)

raw = verification_metrics(paired["observed"], paired["modelled"])
adjusted = verification_metrics(paired["observed"], paired["adjusted"])

print(f"{'':<12}{'bias':>8}{'rmse':>8}")
print(f"{'raw':<12}{raw['bias']:>8.2f}{raw['rmse']:>8.2f}")
print(f"{'adjusted':<12}{adjusted['bias']:>8.2f}{adjusted['rmse']:>8.2f}")

A constant lapse rate assumes a well-mixed atmosphere. During the persistent
winter valley inversions that this study is largely about, temperature
*increases* with height and this correction has the wrong sign -- so report
adjusted and unadjusted side by side rather than replacing one with the other.

## Verification by elevation band

In [ ]:
bands = elevation_band_metrics(per_station)
bands

Empty bands are retained deliberately. A high-elevation band with no stations is
a finding about the observing network, not something to average away -- and it is
precisely the band where Alpine wind farms are increasingly sited.